<a href="https://colab.research.google.com/github/Sirye8/libtorrent-script/blob/main/libtorrent_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Setup and Mount Google Drive
# Install necessary libraries
!apt-get install -qq python3-libtorrent
!python -m pip install --upgrade -q pip setuptools wheel ipython-autotime
!python -m pip install -q lbry-libtorrent

import libtorrent as lt
import time
import sys
import os
from google.colab import drive
from IPython.display import display, clear_output

%load_ext autotime

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}. Please authorize access.")
    sys.exit()

# Define save path in Google Drive
save_path = '/content/drive/MyDrive/TorrentDownloads'
os.makedirs(save_path, exist_ok=True)
print(f"Files will be saved to: {save_path}")

# @title Initialize Torrent Session
ses = lt.session()
ses.listen_on(6881, 6891) # Standard BitTorrent ports
print("Libtorrent session initialized.")

# Enable DHT and Peer Exchange for better peer discovery
ses.add_extension('ut_pex')
ses.add_extension('ut_metadata')
ses.add_extension('metadata_transfer')
ses.add_dht_router("router.utorrent.com", 6881)
ses.add_dht_router("router.bittorrent.com", 6881)
ses.add_dht_router("dht.transmissionbt.com", 6881)
ses.add_dht_router("dht.libtorrent.org", 25401)
ses.start_dht()
print("DHT and PEX enabled.")

# Apply session settings
settings = ses.get_settings()
settings['user_agent'] = 'GoogleColab/TorrentClient/2.1' # Updated user agent
settings['download_rate_limit'] = 0 # Unlimited
settings['upload_rate_limit'] = 0   # Unlimited
settings['connections_limit'] = 200
settings['alert_mask'] = (
    lt.alert.category_t.progress_notification
    | lt.alert.category_t.status_notification
    | lt.alert.category_t.error_notification
    | lt.alert.category_t.tracker_notification
    | lt.alert.category_t.dht_notification
    | lt.alert.category_t.storage_notification
)
ses.apply_settings(settings)
print("Session settings applied.")

state_description = [
    'queued', 'checking_files', 'downloading_metadata', 'downloading',
    'finished', 'seeding', 'allocating', 'checking_resume_data'
]

# --- Helper function for generating progress bar ---
# MODIFIED: Increased default length of progress bar
def generate_progress_bar(percentage, length=40, fill_char='█', empty_char='░'):
    """Generates a text-based progress bar string."""
    if not 0 <= percentage <= 100:
        percentage = 0 if percentage < 0 else 100 # Clamp percentage
    filled_length = int(length * percentage // 100)
    bar = fill_char * filled_length + empty_char * (length - filled_length)
    return f"[{bar}]"

# @title Enter Magnet Link, Select Files, and Download

magnet_link = input("Paste magnet link and press Enter: ")
if not magnet_link.startswith("magnet:?xt=urn:btih:"):
    print("Invalid magnet link format. It should start with 'magnet:?xt=urn:btih:'.")
    sys.exit()

print("\nAdding torrent via magnet URI...")
params = {
    'save_path': save_path,
    'storage_mode': lt.storage_mode_t.storage_mode_sparse,
    'file_priorities': [0] * 2000, # Initially set all to 0 to only fetch metadata
    'flags': (
        lt.add_torrent_params_flags_t.flag_apply_ip_filter |
        lt.add_torrent_params_flags_t.flag_update_subscribe
    )
}

try:
    handle = lt.add_magnet_uri(ses, magnet_link, params)
    print("Torrent added, attempting to fetch metadata (not paused initially).")
except Exception as e:
    print(f"Error adding torrent via magnet URI: {e}")
    sys.exit()

print("Fetching metadata...")
metadata_fetch_timeout = 180  # seconds (3 minutes)
metadata_start_time = time.time()
metadata_fetched_successfully_flag = False # Internal flag for early loop break

while not handle.has_metadata():
    current_time_loop = time.time()
    if current_time_loop - metadata_start_time > metadata_fetch_timeout:
        print(f"\nMetadata fetch timed out after {metadata_fetch_timeout} seconds.")
        print("This could be due to a dead/unpopular torrent, or network issues/restrictions in the Colab environment.")
        if handle.is_valid():
            print("Removing torrent due to metadata timeout.")
            ses.remove_torrent(handle)
        sys.exit() # Exit on timeout

    try:
        alerts = ses.pop_alerts()
        for alert in alerts:
            if isinstance(alert, lt.metadata_received_alert):
                print(f"Alert: Metadata received for torrent!")
                metadata_fetched_successfully_flag = True # Set flag on alert
                break # Break from alert processing loop
            elif isinstance(alert, lt.metadata_failed_alert):
                print(f"Alert: Metadata fetch FAILED: {alert.error.message()}")
                if handle.is_valid():
                    print("Removing torrent due to metadata failure.")
                    ses.remove_torrent(handle)
                sys.exit() # Exit on explicit failure

        if metadata_fetched_successfully_flag:
            if handle.has_metadata(): # Ensure handle has processed it
                break
            # else, continue loop, alert might have been for something else or metadata not fully processed by handle yet

        clear_output(wait=True)
        status = handle.status()
        elapsed_meta_time = int(current_time_loop - metadata_start_time)

        current_status_str = "Unknown"
        if 0 <= status.state < len(state_description):
            current_status_str = state_description[status.state]

        status_line = f"Fetching metadata... State: {current_status_str} (Elapsed: {elapsed_meta_time}s / {metadata_fetch_timeout}s)"

        session_status = ses.status()
        if hasattr(session_status, 'dht_nodes') and session_status.dht_nodes is not None:
             status_line += f" | DHT Nodes: {session_status.dht_nodes}"
        else:
             status_line += f" | DHT Nodes: N/A"

        if status.num_peers >= 0 :
             status_line += f" | Connected Peers: {status.num_peers}"
        print(status_line)

        time.sleep(1)

    except KeyboardInterrupt:
        print("\nOperation cancelled by user during metadata fetch.")
        if handle.is_valid(): ses.remove_torrent(handle)
        sys.exit()
    except Exception as e:
        print(f"\nError during metadata fetch loop: {e}")
        if handle.is_valid(): ses.remove_torrent(handle)
        sys.exit()


if not handle.has_metadata():
    print("Critical: Metadata fetch loop completed, but torrent does not have metadata. (Unexpected: LEXIT_NO_META)")
    if handle.is_valid():
        print("Removing torrent due to lack of metadata post-loop.")
        ses.remove_torrent(handle)
    sys.exit()

print("Metadata received successfully!")

if not handle.is_valid():
    print("Critical: Torrent handle became invalid after metadata was confirmed. (HNDL_INVALID_POST_CONFIRM)")
    sys.exit()

print("Pausing torrent before file selection...")
handle.pause(flags=lt.torrent_handle.graceful_pause)
time.sleep(1) # Give it a moment to pause
status_after_pause = handle.status()
if status_after_pause.paused:
    print("Torrent successfully paused.")
else:
    print("Warning: Torrent may not have paused properly. Continuing with file selection.")


torrent_info = handle.get_torrent_info()
if not torrent_info:
    print("Error: Could not get torrent_info object even after metadata received and confirmed.")
    if handle.is_valid(): ses.remove_torrent(handle)
    sys.exit()

# --- File Listing and Selection with Sorting ---
files_object = torrent_info.files() # This is a libtorrent.file_storage object
num_total_files = files_object.num_files()

# 1. Prepare File Information
files_to_display = []
if num_total_files > 0:
    for i in range(num_total_files):
        file_path_val = files_object.file_path(i)
        file_size_val = files_object.file_size(i)

        files_to_display.append({
            'original_index': i,
            'path': file_path_val,
            'size': file_size_val
        })

    files_to_display.sort(key=lambda f: f['path'])

    print("\nFiles in torrent (sorted alphabetically):")
    for displayed_idx, file_info in enumerate(files_to_display):
        size_mb = file_info['size'] / (1024 * 1024)
        print(f"[{displayed_idx}] {file_info['path']} ({size_mb:.2f} MB)")
else:
    print("\nNo files found in this torrent.")

user_selected_original_indices = []
actual_file_priorities = [0] * num_total_files # Initialize once

# --- MODIFIED: Outer loop for confirmation and retry ---
selection_confirmed = False
while not selection_confirmed:
    user_selected_original_indices = [] # Reset for each attempt
    # actual_file_priorities is already initialized with all 0s,
    # it will be repopulated based on new selection.

    if num_total_files == 0:
        print("No files to select or download.")
        selection_confirmed = True # Nothing to confirm, proceed to (empty) download
        break # Exit confirmation loop

    # Inner loop for file selection (mostly unchanged)
    while True:
        try:
            prompt_message = (f"\nEnter displayed indices/ranges of files to download "
                              f"(e.g., '0,2,5' or '0-4,7,10-12'), 'all', or 'none' "
                              f"(Displayed indices 0-{num_total_files-1 if num_total_files > 0 else 0}): ")
            selection_input = input(prompt_message)
            selection_input_lower = selection_input.strip().lower()

            if selection_input_lower == 'all':
                user_selected_original_indices = [f_info['original_index'] for f_info in files_to_display]
                break # Exit inner selection loop
            elif selection_input_lower == 'none':
                user_selected_original_indices = []
                break # Exit inner selection loop
            else:
                raw_parts = [part.strip() for part in selection_input.split(',') if part.strip()]
                if not raw_parts:
                    print("No selection. Please enter file numbers/ranges, 'all', or 'none'.")
                    continue

                parsed_displayed_indices_set = set()
                valid_input_overall = True
                for part in raw_parts:
                    if '-' in part:
                        try:
                            start_str, end_str = part.split('-', 1)
                            start_idx = int(start_str.strip())
                            end_idx = int(end_str.strip())
                            if start_idx > end_idx:
                                print(f"Error: Invalid range '{part}'. Start index ({start_idx}) cannot be greater than end index ({end_idx}).")
                                valid_input_overall = False; break
                            if not (0 <= start_idx < num_total_files and 0 <= end_idx < num_total_files):
                                print(f"Error: Range '{part}' contains indices out of bounds. Valid displayed indices are 0-{num_total_files-1}.")
                                valid_input_overall = False; break
                            for i_range in range(start_idx, end_idx + 1):
                                parsed_displayed_indices_set.add(i_range)
                        except ValueError:
                            print(f"Error: Invalid range format '{part}'. Please use N-M (e.g., 0-4).")
                            valid_input_overall = False; break
                    elif part.isdigit():
                        try:
                            d_idx = int(part)
                            if 0 <= d_idx < num_total_files:
                                parsed_displayed_indices_set.add(d_idx)
                            else:
                                print(f"Error: Displayed index {d_idx} out of range. Valid displayed indices are 0-{num_total_files-1}.")
                                valid_input_overall = False; break
                        except ValueError:
                            print(f"Error: '{part}' is not a valid number.")
                            valid_input_overall = False; break
                    else:
                        print(f"Error: '{part}' is not a valid number or range (N-M).")
                        valid_input_overall = False; break
                if not valid_input_overall:
                    print("Please correct the errors and try again, or enter 'all'/'none'.")
                    continue
                if not parsed_displayed_indices_set and valid_input_overall: # Handles cases like empty string after split if input was just ","
                    print("No valid file indices or ranges were entered.")
                    continue

                temp_original_indices = []
                for d_idx in sorted(list(parsed_displayed_indices_set)): # Iterate over sorted displayed indices
                    temp_original_indices.append(files_to_display[d_idx]['original_index'])
                user_selected_original_indices = sorted(list(set(temp_original_indices))) # Store unique original indices, sorted
                break # Exit inner selection loop
        except ValueError:
            print("Invalid input format. Please use comma-separated numbers, ranges (e.g., 0-4), 'all', or 'none'.")
        except KeyboardInterrupt:
            print("\nOperation cancelled during file selection.")
            if handle.is_valid(): ses.remove_torrent(handle)
            sys.exit()
    # End of inner file selection loop

    # --- Confirmation Step ---
    actual_file_priorities = [0] * num_total_files # Reset priorities for this confirmation attempt
    selected_paths_for_confirmation = []

    if user_selected_original_indices:
        print("\nYou have selected the following files for download:")
        for orig_idx in user_selected_original_indices: # Iterate through the sorted original indices
            if 0 <= orig_idx < num_total_files:
                actual_file_priorities[orig_idx] = 1 # Tentatively set priority
                # Find the path for confirmation message (full path)
                for f_info in files_to_display: # files_to_display is sorted by path
                    if f_info['original_index'] == orig_idx:
                        selected_paths_for_confirmation.append(f_info['path'])
                        break
        for path in sorted(selected_paths_for_confirmation): # Print sorted paths
            print(f"- {path}")

        while True: # Loop for confirmation input
            confirm_choice = input("Proceed with downloading these files? (yes/no/retry): ").strip().lower()
            if confirm_choice in ['yes', 'y']:
                selection_confirmed = True
                break # Exit confirmation input loop
            elif confirm_choice in ['no', 'n']:
                print("Exiting as per user choice.")
                if handle.is_valid(): ses.remove_torrent(handle)
                sys.exit()
            elif confirm_choice in ['retry', 'r']:
                print("Retrying file selection...")
                break # Exit confirmation input loop, will retry outer selection_confirmed loop
            else:
                print("Invalid choice. Please enter 'yes', 'no', or 'retry'.")
    else: # No files selected
        print("\nNo files were selected.")
        while True: # Loop for no-files confirmation
            confirm_choice = input("Do you want to retry selection or exit? (retry/exit): ").strip().lower()
            if confirm_choice in ['retry', 'r']:
                print("Retrying file selection...")
                break # Exit no-files confirmation, will retry outer selection_confirmed loop
            elif confirm_choice in ['exit', 'e']:
                print("Exiting as no files were selected.")
                if handle.is_valid(): ses.remove_torrent(handle)
                sys.exit()
            else:
                print("Invalid choice. Please enter 'retry' or 'exit'.")

    if selection_confirmed: # If 'yes' was chosen for selected files
        break # Exit the main confirmation loop (while not selection_confirmed)
    # If 'retry' was chosen (either for selected files or no files), the main loop continues

# --- End of Confirmation Logic ---


if not user_selected_original_indices and num_total_files > 0:
    # This case should ideally be handled by the confirmation logic leading to an exit or retry.
    # If it's reached, it implies an issue or an edge case not fully covered by retry/exit.
    # For safety, if no files are selected and we somehow bypass confirmation to exit:
    print("Exiting as no files were selected and not confirmed to proceed.")
    if handle.is_valid(): ses.remove_torrent(handle)
    sys.exit()

if num_total_files > 0 and user_selected_original_indices : # Only apply if there are files and some are selected
    handle.prioritize_files(actual_file_priorities)
    print("File priorities applied based on selection.")
elif num_total_files > 0 and not user_selected_original_indices:
    print("No files selected to download. Proceeding to finish (will download nothing).")


print("\nStarting/Resuming download...")
if handle.is_valid():
    handle.resume()
start_time = time.time()
download_loop_completed_normally = False # Flag to indicate if loop finished due to completion

# Main download loop
while True:
    if not handle.is_valid():
        print("\nTorrent handle became invalid during download. Exiting.")
        break

    status = handle.status()
    is_finished_for_selected = False
    total_selected_wanted_bytes = 0
    total_selected_downloaded_bytes = 0

    current_file_progress_bytes = []
    if num_total_files > 0 and user_selected_original_indices:
        # Ensure handle is valid before calling file_progress
        if handle.is_valid():
            current_file_progress_bytes = handle.file_progress(flags=lt.torrent_handle.piece_granularity)
        else: # Handle became invalid, cannot get progress
            print("\nTorrent handle invalid, cannot get file progress. Exiting loop.")
            break


    if not user_selected_original_indices and num_total_files > 0 :
        is_finished_for_selected = True # No files selected, so "finished"
    elif num_total_files == 0:
        is_finished_for_selected = True # No files in torrent, so "finished"
    else: # Files were selected
        for original_idx, priority_val in enumerate(actual_file_priorities):
            if priority_val > 0: # Only consider files marked for download
                file_size_val = files_object.file_size(original_idx)
                total_selected_wanted_bytes += file_size_val
                if original_idx < len(current_file_progress_bytes):
                    total_selected_downloaded_bytes += current_file_progress_bytes[original_idx]

        # Check for completion of selected files
        if total_selected_wanted_bytes == 0 and user_selected_original_indices: # All selected files are 0-byte
            is_finished_for_selected = True
        elif total_selected_wanted_bytes > 0 and total_selected_downloaded_bytes >= total_selected_wanted_bytes:
            # Add a small tolerance (e.g. 1KB) in case of minor discrepancies if needed, but exact match is ideal
            if (total_selected_downloaded_bytes - total_selected_wanted_bytes) >= -1024 : # Allow for slight over-download or rounding
                 is_finished_for_selected = True


    if is_finished_for_selected:
        # This block means all *selected* files are downloaded, or no files were selected/to download.
        # The torrent might still be 'downloading' if other non-prioritized files exist, or 'seeding'.
        # We break here as our primary goal (downloading selected files) is met.
        if (status.state == lt.torrent_status.states.seeding or \
            status.state == lt.torrent_status.states.finished or \
            (total_selected_wanted_bytes == 0 and user_selected_original_indices) or \
            (not user_selected_original_indices and num_total_files > 0) or \
            num_total_files == 0):
            print("\nDownload of selected files complete, or no files/data to download.")
            download_loop_completed_normally = True
            break
        print("\nAll selected files appear to be downloaded based on byte count. Finalizing...")
        download_loop_completed_normally = True
        break

    try:
        alerts = ses.pop_alerts()
        for alert in alerts:
            if isinstance(alert, lt.torrent_error_alert):
                print(f"\nTorrent Error: {alert.error.message()}")
            elif isinstance(alert, lt.save_resume_data_failed_alert):
                print(f"\nFailed to save resume data: {alert.error.message()}")
            elif isinstance(alert, lt.file_completed_alert):
                # Check if the completed file is one of the selected files
                if alert.index < len(actual_file_priorities) and actual_file_priorities[alert.index] > 0:
                    # Find the path of the completed file using files_to_display for sorted name
                    completed_file_path = "Unknown File"
                    for f_info in files_to_display:
                        if f_info['original_index'] == alert.index:
                            completed_file_path = f_info['path']
                            break
                    print(f"\nSelected file completed: {completed_file_path}")


        clear_output(wait=True)
        torrent_name_str = torrent_info.name() if torrent_info and torrent_info.name() else "N/A"
        print(f"Torrent: {torrent_name_str} (Save: {save_path})")

        progress_display_percent = 0.0
        if total_selected_wanted_bytes > 0:
            progress_display_percent = (total_selected_downloaded_bytes / total_selected_wanted_bytes) * 100
        elif (not user_selected_original_indices and num_total_files > 0) or \
             (total_selected_wanted_bytes == 0 and user_selected_original_indices) or \
             num_total_files == 0: # Handles cases where nothing is to be downloaded
            progress_display_percent = 100.0

        # Overall progress bar (uses new default length from generate_progress_bar)
        overall_progress_bar = generate_progress_bar(progress_display_percent)
        print(f"Overall Progress (Selected): {overall_progress_bar} {progress_display_percent:.2f}%")

        current_state_str = "Unknown"
        if 0 <= status.state < len(state_description):
            current_state_str = state_description[status.state]
        print(f"Status: {current_state_str}")

        if user_selected_original_indices and num_total_files > 0: # Only show if files were selected
            print("-" * 70) # Separator, wider for potentially longer names
            print("Individual File Progress:")
            # Iterate through files_to_display to show them in sorted order
            for f_info_display in files_to_display:
                original_idx = f_info_display['original_index']
                if actual_file_priorities[original_idx] > 0: # Only show selected files
                    file_name = f_info_display['path'] # Full file name from sorted list
                    total_size = f_info_display['size']
                    downloaded_size = 0
                    if original_idx < len(current_file_progress_bytes): # Check bounds
                        downloaded_size = current_file_progress_bytes[original_idx]

                    file_percent = 0.0
                    if total_size > 0:
                        file_percent = (downloaded_size / total_size) * 100
                    elif total_size == 0: # Handle 0-byte files as 100% if selected
                        file_percent = 100.0

                    # MODIFIED: Increased length for individual file progress bar
                    file_progress_bar = generate_progress_bar(file_percent, length=30)
                    downloaded_mb = downloaded_size / (1024*1024)
                    total_mb = total_size / (1024*1024)
                    # Display full file name without truncation
                    print(f"  - {file_name}")
                    print(f"    {file_progress_bar} {downloaded_mb:.2f}/{total_mb:.2f}MB ({file_percent:.1f}%)")
            print("-" * 70) # Separator

        session_status_dl = ses.status()
        dht_nodes_count = session_status_dl.dht_nodes if hasattr(session_status_dl, 'dht_nodes') and session_status_dl.dht_nodes is not None else "N/A"
        print(f"Peers: {status.num_peers} (Seeds: {status.num_seeds}, Conns: {status.num_connections}, DHT: {dht_nodes_count})")
        dl_speed_bytes = status.download_payload_rate
        ul_speed_bytes = status.upload_payload_rate
        dl_speed_kb = dl_speed_bytes / 1024
        ul_speed_kb = ul_speed_bytes / 1024
        dl_unit, ul_unit = "KB/s", "KB/s"
        if dl_speed_kb > 1024: dl_speed_kb /= 1024; dl_unit = "MB/s"
        if ul_speed_kb > 1024: ul_speed_kb /= 1024; ul_unit = "MB/s"
        print(f"DL Speed: {dl_speed_kb:.2f} {dl_unit} | UL Speed: {ul_speed_kb:.2f} {ul_unit}")
        print(f"Total Selected Downloaded: {total_selected_downloaded_bytes / (1024*1024):.2f} MB / {total_selected_wanted_bytes / (1024*1024):.2f} MB")

        if dl_speed_bytes > 0 and total_selected_wanted_bytes > total_selected_downloaded_bytes:
            remaining_sel_bytes = total_selected_wanted_bytes - total_selected_downloaded_bytes
            if remaining_sel_bytes > 0 : # Ensure remaining is positive before ETA calc
                eta_seconds = remaining_sel_bytes / dl_speed_bytes
                print(f"ETA (Selected): {time.strftime('%H:%M:%S', time.gmtime(eta_seconds))}")
            else: # Should be caught by is_finished_for_selected, but as a fallback
                print("ETA (Selected): Finalizing...")
        elif total_selected_downloaded_bytes >= total_selected_wanted_bytes and total_selected_wanted_bytes > 0:
            print("ETA (Selected): Done!")
        elif (not user_selected_original_indices and num_total_files > 0) or \
             (total_selected_wanted_bytes == 0 and user_selected_original_indices) or \
             num_total_files == 0: # No actual download happening or needed
             print("ETA (Selected): N/A")
        else: # Still downloading or calculating
            print("ETA (Selected): Calculating...")

        print(f"Elapsed Time: {time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))}")
        time.sleep(1)

    except KeyboardInterrupt:
        print("\nDownload interrupted by user.")
        if handle.is_valid():
            handle.pause(flags=lt.torrent_handle.graceful_pause)
            print("Attempting to save resume data...")
            handle.save_resume_data(flags=lt.save_resume_flags_t.flush_disk_cache)
            resume_data_saved = False
            save_resume_start_time = time.time()
            while time.time() - save_resume_start_time < 10: # Wait up to 10s for alert
                alerts = ses.pop_alerts()
                for alert in alerts:
                    if isinstance(alert, lt.save_resume_data_alert):
                        print("Resume data saved successfully.")
                        resume_data_saved = True; break
                    elif isinstance(alert, lt.save_resume_data_failed_alert):
                        print(f"Failed to save resume data: {alert.error.message()}")
                        resume_data_saved = True; break # Treat as handled
                if resume_data_saved: break
                time.sleep(0.2)
            if not resume_data_saved: print("Did not receive resume data saved/failed alert in time.")
            ses.remove_torrent(handle)
        print("Torrent removed from session.")
        sys.exit()
    except Exception as e:
        print(f"\nError during download loop: {e}")
        if handle.is_valid(): handle.pause(); ses.remove_torrent(handle) # Basic pause on error
        sys.exit()

# --- After Download Loop ---
clear_output(wait=True) # Clear the last status update
elapsed_total_time = time.time() - start_time
print("="*30); print(" Download Process Ended "); print("="*30)

final_torrent_name = "N/A"
# Try to get torrent name, handle might be invalid if removed due to error earlier
current_torrent_info = None
if handle.is_valid():
    current_torrent_info = handle.get_torrent_info()
if current_torrent_info and current_torrent_info.name():
    final_torrent_name = current_torrent_info.name()
elif torrent_info and torrent_info.name(): # Fallback to initially fetched torrent_info if handle became invalid
     final_torrent_name = torrent_info.name()


print(f"Torrent: {final_torrent_name}")
print(f"Saved to: {save_path}")
print(f"Total Time: {time.strftime('%H:%M:%S', time.gmtime(elapsed_total_time))}")

# Recalculate final downloaded/wanted bytes for selected files
final_sel_wanted_bytes, final_sel_downloaded_bytes = 0, 0
download_successful_for_selected_files = False

if handle.is_valid() and files_object and num_total_files > 0 and user_selected_original_indices:
    final_file_progress_bytes = handle.file_progress(flags=lt.torrent_handle.piece_granularity)
    for original_idx, priority_val in enumerate(actual_file_priorities):
        if priority_val > 0: # Only selected files
            file_s = files_object.file_size(original_idx)
            final_sel_wanted_bytes += file_s
            if original_idx < len(final_file_progress_bytes):
                final_sel_downloaded_bytes += final_file_progress_bytes[original_idx]

if num_total_files > 0 and user_selected_original_indices:
    final_overall_percent = 0.0
    if final_sel_wanted_bytes > 0:
        final_overall_percent = (final_sel_downloaded_bytes / final_sel_wanted_bytes) * 100
    elif final_sel_wanted_bytes == 0 : # All selected files were 0-byte
        final_overall_percent = 100.0

    # Final overall progress bar (uses new default length)
    final_overall_bar = generate_progress_bar(final_overall_percent)
    print(f"Final Overall Progress: {final_overall_bar} {final_overall_percent:.2f}%")
    print(f"Downloaded (Selected Files): {final_sel_downloaded_bytes / (1024*1024):.2f} MB / {final_sel_wanted_bytes / (1024*1024):.2f} MB")

    # Determine if download was successful for selected files
    if final_sel_wanted_bytes == 0: # All selected files were 0-byte
        download_successful_for_selected_files = True
        print("All selected files (0-byte) processed.")
    elif final_sel_downloaded_bytes >= final_sel_wanted_bytes - 1024: # Allow 1KB tolerance
        download_successful_for_selected_files = True
        print("All selected files downloaded successfully!")
    else:
        print(f"Download may not have completed fully for all selected files.")

elif not user_selected_original_indices and num_total_files > 0:
    print("No files were selected for download.")
    download_loop_completed_normally = True # Considered "normal" as no download was intended
elif num_total_files == 0:
    print("Torrent contained no files.")
    download_loop_completed_normally = True # "Normal" as no files to download

# --- MODIFIED: .parts file deletion logic ---
if download_loop_completed_normally and download_successful_for_selected_files:
    print("\nAttempting to delete .parts file(s) as download of selected files is complete...")
    any_parts_file_deleted_overall = False

    # Check inside a potential subdirectory named after the torrent
    ti_for_parts_check = None
    if handle.is_valid(): # Get current info if handle is still valid
        ti_for_parts_check = handle.get_torrent_info()
    if not ti_for_parts_check and torrent_info: # Fallback to initial torrent_info
        ti_for_parts_check = torrent_info

    if ti_for_parts_check and ti_for_parts_check.name():
        torrent_content_path = os.path.join(save_path, ti_for_parts_check.name())
        if os.path.isdir(torrent_content_path):
            # print(f"DEBUG: Checking for .parts files in subdirectory: {torrent_content_path}")
            try:
                for filename in os.listdir(torrent_content_path):
                    if filename.endswith(".parts"):
                        parts_file_path = os.path.join(torrent_content_path, filename)
                        try:
                            os.remove(parts_file_path)
                            print(f"Successfully deleted .parts file: {parts_file_path}")
                            any_parts_file_deleted_overall = True
                        except OSError as e:
                            print(f"Error deleting .parts file {parts_file_path}: {e}")
            except Exception as e:
                print(f"Error listing directory {torrent_content_path} for .parts file deletion: {e}")

    # Check the root save_path directory
    # print(f"DEBUG: Checking for .parts files in root save directory: {save_path}")
    try:
        for filename in os.listdir(save_path):
            if filename.endswith(".parts"):
                parts_file_path = os.path.join(save_path, filename)
                if os.path.exists(parts_file_path):
                    try:
                        os.remove(parts_file_path)
                        print(f"Successfully deleted .parts file: {parts_file_path}")
                        any_parts_file_deleted_overall = True
                    except OSError as e:
                        print(f"Error deleting .parts file {parts_file_path}: {e}")
    except Exception as e:
        print(f"Error listing root directory {save_path} for .parts file deletion: {e}")

    if not any_parts_file_deleted_overall:
        print(f"No .parts files found in {save_path} (or its torrent subdirectory if applicable) or none could be deleted.")
elif download_loop_completed_normally and not download_successful_for_selected_files and user_selected_original_indices:
     print("\nDownload of selected files was not fully complete. .parts file(s) will not be deleted by this script.")


if handle.is_valid():
    handle.pause(flags=lt.torrent_handle.graceful_pause)
    print("Torrent paused. Requesting save of resume data...")
    handle.save_resume_data(flags=lt.save_resume_flags_t.flush_disk_cache) # Request resume data

    resume_data_final_saved = False
    save_resume_final_start_time = time.time()
    while time.time() - save_resume_final_start_time < 5: # Wait up to 5s for alert
        alerts = ses.pop_alerts()
        for alert in alerts:
            if isinstance(alert, lt.save_resume_data_alert):
                print("Resume data saved successfully before exiting.")
                resume_data_final_saved = True; break
            elif isinstance(alert, lt.save_resume_data_failed_alert):
                print(f"Failed to save resume data before exiting: {alert.error.message()}")
                resume_data_final_saved = True; break # Treat as handled
        if resume_data_final_saved: break
        time.sleep(0.2)
    if not resume_data_final_saved: print("Did not receive final resume data saved/failed alert in time.")

    ses.remove_torrent(handle)
    print("Torrent removed from session.")
else:
    print("Torrent handle was invalid at script end (likely already removed due to error or prior completion).")

print("\nScript finished.")